# STEP 1.0 数据审计与事件队列
本模块只读载入更新工作簿，生成可复核事件编号、来源等级、事件类型和核心/扩展分析队列。

In [1]:
from pathlib import Path # 导入跨平台路径工具
import sys # 导入解释器路径模块
ROOT=Path.cwd().resolve() # 读取当前工作目录
ROOT=ROOT.parent if ROOT.name=='notebooks' else ROOT # 从notebooks目录回到项目根目录
sys.path.insert(0,str(ROOT)) # 将项目根目录加入模块搜索路径
from src.pipeline import prepare_events,build_case_control_calendar,write_audit # 导入事件审计与病例交叉函数
SOURCE=ROOT/'data/raw/全球城市高层建筑火灾事件数据库_2000-2026.xlsx' # 指定更新工作簿快照
PROCESSED=ROOT/'data/processed/events_standardised.csv' # 指定标准化事件表输出路径
CALENDAR=ROOT/'data/interim/case_control_calendar.csv' # 指定病例交叉日历输出路径

In [2]:
events=prepare_events(SOURCE) # 标准化更新事件数据库并构造队列
events.to_csv(PROCESSED,index=False,encoding='utf-8-sig') # 保存机器可读标准化事件表
calendar=build_case_control_calendar(events) # 构造同月同星期病例交叉日期
calendar.to_csv(CALENDAR,index=False,encoding='utf-8-sig') # 保存病例与对照日历
audit=write_audit(events,ROOT/'outputs/tables/event_audit.json') # 保存可复核数据审计报告
print(audit) # 显示核心审计结果
print(events['cohort_reason'].value_counts(dropna=False)) # 显示队列纳入与排除原因

{'events_total': 239, 'geocoded': 233, 'analysis_core': 191, 'analysis_extended': 228, 'external_trigger': 5, 'arson_trigger': 8, 'construction_related': 24, 'possible_duplicate': 0, 'countries': 52, 'date_min': '2000-08-02', 'date_max': '2026-08-23', 'source_sha256': '386eb556101286e4dce89060d1d60ca105f5f54e97db3d3f297bcc2be6c291e5'}
cohort_reason
core_eligible                 191
compendium_only_source         37
invalid_or_missing_geocode      6
external_disaster_trigger       5
Name: count, dtype: int64
